# BayMorph — Recovering a Simulated Galaxy

This notebook is the controlled end-to-end BayMorph example. It creates
a noisy single-Sérsic galaxy with known parameters, asks which inference
sampler to use, writes a dedicated YAML configuration, runs the public
workflow, and compares the posterior with the known truth.

The sampler is a numerical choice: the simulated observation, priors,
likelihood, parameter meanings, and result container remain the same.
Nautilus, Dynesty, and pocoMC estimate Bayesian evidence. emcee is a
posterior-only ensemble-MCMC backend and therefore returns no `logZ` or
Bayes factor.

Run the cells from top to bottom. The numerical settings are transparent
tutorial starting points, not publication-quality convergence claims.


In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import yaml
from astropy.table import Table

from baymorph import BayMorphRunner
from baymorph.plotting import plot_log_image


## 1. Choose the inference sampler

The choice is made before the YAML is written. Enter exactly one key:

| Key | Method | Scientific product |
| --- | --- | --- |
| `nautilus` | Neural-assisted nested sampling | Posterior and evidence |
| `dynesty-static` | Static nested sampling | Posterior and evidence |
| `dynesty-dynamic` | Dynamic nested sampling | Posterior and evidence |
| `pocomc` | Sequential Monte Carlo | Posterior and evidence |
| `emcee` | Ensemble MCMC | Posterior only |

emcee additionally asks whether this is a new (`fresh`) chain or a
continuation (`resume`). Its `max_steps` value is the total production
target, not the number of new steps. A fresh run uses two worker
processes here so the simulated example also exercises parallel
likelihood evaluation. The YAML Dynesty presets intentionally omit
`rstate`, which is a NumPy runtime object rather than a portable YAML
value; repeated Dynesty tutorial runs are not guaranteed to be bitwise
identical.


In [ ]:
sampler_descriptions = {
    "nautilus": "posterior + evidence",
    "dynesty-static": "posterior + evidence",
    "dynesty-dynamic": "posterior + evidence",
    "pocomc": "posterior + evidence",
    "emcee": "posterior only; no evidence",
}

print("Available samplers:")
for key, description in sampler_descriptions.items():
    print(f"  {key:16s} {description}")

sampler_choice = input("Choose one sampler: ").strip().lower()

if sampler_choice not in sampler_descriptions:
    raise ValueError(
        "Sampler must be one of: "
        + ", ".join(sampler_descriptions)
    )

checkpoint_mode = None
emcee_max_steps = None

if sampler_choice == "emcee":
    checkpoint_mode = (
        input("emcee checkpoint mode [fresh/resume, default fresh]: ")
        .strip()
        .lower()
        or "fresh"
    )

    if checkpoint_mode not in {"fresh", "resume"}:
        raise ValueError(
            "emcee checkpoint mode must be 'fresh' or 'resume'."
        )

    default_steps = 1000 if checkpoint_mode == "fresh" else 3000
    requested_steps = input(
        "Total emcee production steps "
        f"[default {default_steps}]: "
    ).strip()
    emcee_max_steps = int(requested_steps or default_steps)

    if emcee_max_steps < 1:
        raise ValueError("emcee max_steps must be positive.")

print(f"Selected sampler: {sampler_choice}")


## 2. Build and review the YAML

BayMorph's shipped `configs/simulated_galaxy.yaml` remains an immutable
scientific template. The cell below changes only the sampler envelope
and the run identity, then prints the complete generated YAML.

Each sampler receives a separate object name, FITS path, run directory,
and configuration file. Results from different backends therefore never
overwrite one another. Model comparison is disabled because this
notebook fits one model; it is also unavailable for emcee because emcee
does not estimate evidence.


In [ ]:
def tutorial_sampler_config(choice):
    # Return one explicit small-run sampler configuration.
    serial_execution = {
        "processes": 1,
        "progress": True,
    }

    configurations = {
        "nautilus": {
            "name": "nautilus",
            "execution": serial_execution,
            "init": {
                "n_live": 100,
                "seed": 42,
            },
            "run": {
                "n_eff": 200,
                "f_live": 0.05,
            },
        },
        "dynesty-static": {
            "name": "dynesty",
            "mode": "static",
            "execution": serial_execution,
            "init": {
                "nlive": 100,
                "sample": "rwalk",
                "bound": "multi",
            },
            "run": {
                "dlogz": 0.1,
            },
        },
        "dynesty-dynamic": {
            "name": "dynesty",
            "mode": "dynamic",
            "execution": serial_execution,
            "init": {
                "sample": "rwalk",
                "bound": "multi",
            },
            "run": {
                "nlive_init": 100,
                "nlive_batch": 50,
                "dlogz_init": 0.1,
            },
        },
        "pocomc": {
            "name": "pocomc",
            "execution": serial_execution,
            "init": {
                "n_effective": 256,
                "n_active": 128,
                "random_state": 42,
            },
            "run": {
                "n_total": 1024,
                "n_evidence": 1024,
            },
        },
        "emcee": {
            "name": "emcee",
            "execution": {
                "processes": 2,
                "progress": True,
            },
            "init": {
                "nwalkers": "auto",
                "seed": 42,
                "checkpoint_path": "auto",
                "checkpoint_mode": checkpoint_mode,
            },
            "run": {
                "burn_in": 300,
                "max_steps": emcee_max_steps,
                "check_interval": 100,
                "min_autocorr_times": 50.0,
                "tau_tolerance": 0.05,
            },
        },
    }

    return deepcopy(configurations[choice])


template_request = Path("configs/simulated_galaxy.yaml")
template_runner = BayMorphRunner(
    template_request,
    verbose=False,
)
template_path = template_runner.config_path

if template_path is None:
    raise RuntimeError("Could not resolve the simulation template.")

selected_config = yaml.safe_load(
    template_path.read_text(encoding="utf-8")
)

run_key = sampler_choice.replace("-", "_")
analysis_name = f"simulated_galaxy_{run_key}"
config_path = template_path.parent / f"{analysis_name}.yaml"

selected_config["object"]["name"] = analysis_name
selected_config["data"]["image_path"] = (
    f"runs/{analysis_name}/input/{analysis_name}.fits"
)
selected_config["fit"]["sampler"] = tutorial_sampler_config(
    sampler_choice
)
selected_config["fit"]["comparison"] = {
    "enabled": False,
    "metric": "bayes_factor",
}

yaml_text = yaml.safe_dump(
    selected_config,
    sort_keys=False,
)

print(f"Generated configuration: {config_path}")
print()
print(yaml_text)


## 3. Write the reviewed YAML

Run the next cell only after checking the printed truth, priors,
likelihood, sampler settings, and output paths. Typing `write` is the
explicit approval to create the new configuration; the template itself
is never overwritten.


In [ ]:
confirmation = input(
    f"Type 'write' to create {config_path.name}: "
).strip().lower()

if confirmation != "write":
    raise RuntimeError("YAML writing was not confirmed.")

config_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)
config_path.write_text(
    yaml_text,
    encoding="utf-8",
)

print(f"Wrote {config_path}")


## 4. Run the complete analysis

The all-in-one runner generates the synthetic FITS image, prepares it,
loads the configured analytic Gaussian PSF, performs inference, persists
the result, and creates diagnostics.

If an emcee checkpoint already exists, `fresh` intentionally refuses to
overwrite it. Choose `resume` in step 1 and increase the total
`max_steps` target only when the stored chain is scientifically suitable
for continuation. A checkpoint already marked `converged` adds no further
steps. Independent-seed validation requires a distinct fresh run rather
than resuming the same chain.


In [ ]:
runner = BayMorphRunner(config_path)
fit = runner.run(diagnostics=True)


### Step-by-step equivalent

The all-in-one call above is recommended for normal use. To inspect each
stage in a fresh Python session, use the equivalent public workflow:

    runner = BayMorphRunner(config_path)
    runner.simulate()
    runner.prepare_image()
    fit = runner.fit()
    runner.diagnostics()

The Gaussian PSF is analytic, so this example requires no empirical PSF
construction step. Do not execute both workflows for the same fresh emcee
checkpoint.


## 5. Inspect the simulated observation

This is the noisy image presented to the same preparation and fitting
machinery used for an observed galaxy.


In [ ]:
plot_log_image(
    runner.simulation.image,
    title="Simulated single-Sérsic observation",
    colorbar_label="Counts",
    show=True,
)


## 6. Check parameter recovery

The simulation truth was fixed before inference. The table places those
known values beside the posterior summaries returned by BayMorph.
Angular values are expressed in radians to match the fitted
parameterization.

A short run matching the truth once is a smoke test, not a convergence
validation. Publication use requires sampler-appropriate diagnostics and
repeat runs.


In [ ]:
truth = dict(
    runner.config["simulation"]["truth"]
)
truth["theta"] = np.deg2rad(
    truth.pop("theta_deg")
)

parameters = (
    "flux",
    "re",
    "n",
    "q",
    "theta",
    "x0",
    "y0",
)

recovery_rows = []

for parameter in parameters:
    recovery_rows.append(
        {
            "parameter": parameter,
            "truth": truth[parameter],
            **fit.posterior.summary(parameter),
        }
    )

recovery_table = Table(
    rows=recovery_rows
)
recovery_table


## 7. Inspect the backend-neutral result

Every backend returns posterior samples, diagnostics, provenance, fit
statistics, best-fitting parameters, and persisted products. Evidence is
present only when the selected algorithm estimates it.

For emcee, inspect convergence, autocorrelation time, acceptance
fractions, MCMC effective sample size, trace stability, and agreement
across independent seeds. Equal posterior weights do not by themselves
imply independent samples.


In [ ]:
inference = fit.posterior
fit_summary = fit.summary()
backend_name = inference.provenance.get("backend", "unknown")

print(f"Backend: {backend_name}")

if inference.evidence is None:
    print("Evidence: unavailable (posterior-only backend)")
    print(
        "MCMC effective sample size:",
        inference.effective_sample_size,
    )
    print(
        "Converged:",
        inference.diagnostics.get("converged"),
    )
else:
    print(f"logZ: {inference.logz}")
    print(f"logZ uncertainty: {inference.logz_error}")

fit_summary
